In [2]:
import pickle

# 读取 .pkl 文件
with open('results.pkl', 'rb') as f:  # 必须用二进制模式 'rb'
    result_dict = pickle.load(f)

In [3]:
import numpy as np
from scipy import stats
import pandas as pd

# Calculate average RMSE and MAE for each model
model_metrics = {}

for model_type in result_dict:
    for task_type in result_dict[model_type]:
        for model_name in result_dict[model_type][task_type]:
            rmses = []
            maes = []
            for idx in result_dict[model_type][task_type][model_name]:
                rmses.append(result_dict[model_type][task_type][model_name][idx]['RMSE'])
                maes.append(result_dict[model_type][task_type][model_name][idx]['MAE'])
            
            full_name = f"{model_type}_{task_type}_{model_name}"
            model_metrics[full_name] = {
                'mean_RMSE': np.mean(rmses),
                'mean_MAE': np.mean(maes),
                'all_RMSE': rmses,
                'all_MAE': maes
            }

# Create DataFrame for average performance
avg_perf_data = []
for model in model_metrics:
    avg_perf_data.append({
        'Model': model,
        'Avg RMSE': model_metrics[model]['mean_RMSE'],
        'Avg MAE': model_metrics[model]['mean_MAE']
    })
    
avg_perf_df = pd.DataFrame(avg_perf_data).sort_values('Avg RMSE')
print("\nAverage Performance Across All Models:")
print(avg_perf_df.to_string(index=False))

# Get all model names
models = list(model_metrics.keys())
n_models = len(models)

# Initialize comparison results storage
pairwise_data = []

# Perform pairwise comparisons
for i in range(n_models):
    for j in range(i+1, n_models):
        model1 = models[i]
        model2 = models[j]
        
        # RMSE comparison
        rmse1 = model_metrics[model1]['all_RMSE']
        rmse2 = model_metrics[model2]['all_RMSE']
        
        # MAE comparison
        mae1 = model_metrics[model1]['all_MAE']
        mae2 = model_metrics[model2]['all_MAE']
        
        # Calculate win/loss counts
        rmse_better = sum(r1 < r2 for r1, r2 in zip(rmse1, rmse2))
        rmse_worse = sum(r1 > r2 for r1, r2 in zip(rmse1, rmse2))
        
        mae_better = sum(m1 < m2 for m1, m2 in zip(mae1, mae2))
        mae_worse = sum(m1 > m2 for m1, m2 in zip(mae1, mae2))
        
        # One-sided t-tests
        rmse_t, rmse_p = stats.ttest_rel(rmse1, rmse2, alternative='less')
        mae_t, mae_p = stats.ttest_rel(mae1, mae2, alternative='less')
        
        pairwise_data.append({
            'Model 1': model1,
            'Model 2': model2,
            'RMSE: Model 1 Better': rmse_better,
            'RMSE: Model 1 Worse': rmse_worse,
            'RMSE p-value': rmse_p,
            'RMSE Significant': rmse_p < 0.05,
            'MAE: Model 1 Better': mae_better,
            'MAE: Model 1 Worse': mae_worse,
            'MAE p-value': mae_p,
            'MAE Significant': mae_p < 0.05
        })

# Create pairwise comparison DataFrame
pairwise_df = pd.DataFrame(pairwise_data)

# Display significant results
significant_results = pairwise_df[
    (pairwise_df['RMSE Significant']) | 
    (pairwise_df['MAE Significant'])
].sort_values(['RMSE p-value', 'MAE p-value'])

print("\nPairwise Model Comparisons (Significant Differences Only):")
print(significant_results.to_string(index=False))

# Optional: Save results to CSV
# avg_perf_df.to_csv('model_average_performance.csv', index=False)
# pairwise_df.to_csv('model_pairwise_comparisons.csv', index=False)
# significant_results.to_csv('significant_differences.csv', index=False)


Average Performance Across All Models:
                                                                    Model  Avg RMSE  Avg MAE
                      FNO2d_recurrent_modes64_width60_epochs500_Tin10_T10  0.311532 0.213168
                      FNO2d_recurrent_modes16_width60_epochs500_Tin10_T10  0.315702 0.216745
                      FNO2d_recurrent_modes48_width60_epochs500_Tin10_T10  0.325217 0.223623
                      FNO2d_recurrent_modes32_width40_epochs500_Tin10_T10  0.335240 0.234046
                      FNO2d_recurrent_modes64_width40_epochs500_Tin10_T10  0.337644 0.235757
                      FNO2d_recurrent_modes32_width60_epochs500_Tin10_T10  0.340296 0.230033
                      FNO2d_recurrent_modes96_width40_epochs500_Tin10_T10  0.341112 0.237695
                      FNO2d_recurrent_modes48_width40_epochs500_Tin10_T10  0.343496 0.239848
                     FNO2d_recurrent_modes128_width40_epochs500_Tin10_T10  0.352749 0.244336
                      FNO2d_re

In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
from matplotlib import cm
import re

# Assuming avg_perf_df is your original DataFrame
# First, let's create a function to parse the model names
def parse_model_name(model_name):
    # Initialize a dictionary to store parsed components
    components = {
        'model_type': None,  # 2d or 3d
        'architecture': None,  # recurrent, initial_to_final
        'modes1': None,
        'modes2': None,
        'modes3': None,
        'width': None,
        'epochs': None,
        'padding': None,  # padding or no_padding (for 3D models)
        'Tin': None,
        'Tout': None,
        # 'full_model_name': model_name  # Keep the original name in a new column
    }
    
    # Extract FNO dimension (2d or 3d)
    if 'FNO2d' in model_name:
        components['model_type'] = '2d'
    elif 'FNO3d' in model_name:
        components['model_type'] = '3d'
    
    # Extract architecture type
    if 'recurrent' in model_name:
        components['architecture'] = 'recurrent'
    elif 'initial_to_final' in model_name:
        components['architecture'] = 'initial_to_final'
    
    # Extract padding info for 3D models
    if 'no_padding' in model_name:
        components['padding'] = 'no'
    elif 'padding' in model_name:
        components['padding'] = 'yes'
    
    # Extract modes information
    if components['model_type'] == '2d':
        # For 2D models, modes parameter applies to both modes1 and modes2
        modes_match = re.search(r'modes(\d+)', model_name)
        if modes_match:
            mode_val = int(modes_match.group(1))
            components['modes1'] = mode_val
            components['modes2'] = mode_val
    elif components['model_type'] == '3d':
        # For 3D models, we have modes12 and modes3
        modes12_match = re.search(r'modes12(\d+)', model_name)
        modes3_match = re.search(r'modes3(\d+)', model_name)
        
        if modes12_match:
            mode_val = int(modes12_match.group(1))
            components['modes1'] = mode_val
            components['modes2'] = mode_val
        if modes3_match:
            components['modes3'] = int(modes3_match.group(1))
    
    # Extract width
    width_match = re.search(r'width(\d+)', model_name)
    if width_match:
        components['width'] = int(width_match.group(1))
    
    # Extract epochs
    epochs_match = re.search(r'epochs(\d+)', model_name)
    if epochs_match:
        components['epochs'] = int(epochs_match.group(1))
    
    # Extract Tin and Tout as integers
    tin_match = re.search(r'Tin(\d+)', model_name)
    if tin_match:
        components['Tin'] = int(tin_match.group(1))
    
    tout_match = re.search(r'_T(\d+)', model_name)  # Added underscore to avoid matching Tin
    if tout_match:
        components['Tout'] = int(tout_match.group(1))
    
    return components

# Create a copy of the original DataFrame
styled_df = avg_perf_df.copy()

# Parse model names and add new columns
parsed_data = styled_df['Model'].apply(parse_model_name).apply(pd.Series)
styled_df = pd.concat([styled_df, parsed_data], axis=1)

# For 2D models, ensure modes1 and modes2 are always the same
styled_df.loc[styled_df['model_type'] == '2d', 'modes2'] = styled_df.loc[styled_df['model_type'] == '2d', 'modes1']

# Replace None/NaN with empty strings
styled_df = styled_df.fillna('')

# Convert numeric columns to proper types (after filling NaN)
numeric_cols = ['modes1', 'modes2', 'modes3', 'width', 'epochs', 'Tin', 'Tout']
for col in numeric_cols:
    styled_df[col] = pd.to_numeric(styled_df[col], errors='ignore')

# Change Model column to just "FNO"
styled_df['Model'] = 'FNO'

# Reorder columns for better readability
column_order = [
    'Model', 'model_type', 'architecture', 'padding',
    'modes1', 'modes2', 'modes3', 'width', 'epochs',
    'Tin', 'Tout', 
     # 'full_model_name', 
    'Avg RMSE', 'Avg MAE'
]
styled_df = styled_df[column_order]

# Now apply the styling as before
# Define color maps for each column
cmap_rmse = cm.viridis
cmap_mae = cm.viridis

# Normalize the data for coloring
norm_rmse = plt.Normalize(styled_df['Avg RMSE'].min(), styled_df['Avg RMSE'].max())
norm_mae = plt.Normalize(styled_df['Avg MAE'].min(), styled_df['Avg MAE'].max())

# Function to apply colors
def color_columns(val, col_name, cmap, norm):
    if val == '':
        return ''
    color = plt.cm.get_cmap(cmap)(norm(val))
    return f'background-color: rgba({color[0]*255:.0f}, {color[1]*255:.0f}, {color[2]*255:.0f}, {color[3]:.1f})'

# Custom formatter function
def format_cell(x):
    if isinstance(x, (int, float)) and not pd.isna(x):
        if x == int(x):
            return f'{int(x)}'
        else:
            return f'{x:.6f}'
    else:
        return ''

# Apply styling
styled_df_style = styled_df.style\
    .apply(lambda x: [color_columns(v, 'Avg RMSE', cmap_rmse, norm_rmse) for v in x], subset=['Avg RMSE'])\
    .apply(lambda x: [color_columns(v, 'Avg MAE', cmap_mae, norm_mae) for v in x], subset=['Avg MAE'])\
    .format({
        'Avg RMSE': '{:.6f}', 
        'Avg MAE': '{:.6f}',
        'modes1': format_cell,
        'modes2': format_cell,
        'modes3': format_cell,
        'width': format_cell,
        'epochs': format_cell,
        'Tin': format_cell,
        'Tout': format_cell
    })\
    .set_properties(**{'text-align': 'center'})\
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#34495e'), 
                 ('color', 'white'),
                 ('font-weight', 'bold')]
    }])

# Display the styled DataFrame
display(styled_df_style)

/tmp/ipykernel_1042874/2311905802.py:102: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  styled_df[col] = pd.to_numeric(styled_df[col], errors='ignore')
/tmp/ipykernel_1042874/2311905802.py:130: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  color = plt.cm.get_cmap(cmap)(norm(val))
/tmp/ipykernel_1042874/2311905802.py:130: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  color = plt.cm.get_cmap(cmap)(norm(val))


,Model,model_type,architecture,padding,modes1,modes2,modes3,width,epochs,Tin,Tout,Avg RMSE,Avg MAE
10,FNO,2d,recurrent,,64,64,,60,500,10,10,0.311532,0.213168
3,FNO,2d,recurrent,,16,16,,60,500,10,10,0.315702,0.216745
8,FNO,2d,recurrent,,48,48,,60,500,10,10,0.325217,0.223623
5,FNO,2d,recurrent,,32,32,,40,500,10,10,0.335240,0.234046
11,FNO,2d,recurrent,,64,64,,40,500,10,10,0.337644,0.235757
6,FNO,2d,recurrent,,32,32,,60,500,10,10,0.340296,0.230033
12,FNO,2d,recurrent,,96,96,,40,500,10,10,0.341112,0.237695
7,FNO,2d,recurrent,,48,48,,40,500,10,10,0.343496,0.239848
13,FNO,2d,recurrent,,128,128,,40,500,10,10,0.352749,0.244336
2,FNO,2d,recurrent,,12,12,,20,500,10,10,0.405792,0.287911
